# Bearing Capacity with Direct GPT-4o-mini Calls

This notebook now opens with a minimal OpenAI API demo before layering in the full Leaning Tower of Pisa bearing-capacity workflow described in `docs/05-llm/TASK_PROMPT.md`.

### Learning objectives
- Verify a live GPT-4o-mini call with just a few lines of code
- Marshal deterministically validated soil/foundation data via `pydantic`
- Extract parameters using OpenAI structured outputs and cross-check with Terzaghi math
- Produce figures, sensitivity plots, and final JSON summaries ready for reports

In [3]:
!pip install -q openai pydantic chromadb pypdf2 tiktoken python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: /Users/krishna/courses/CE397-Scientific-MachineLearning/utp-sciml/env/bin/python -m pip install --upgrade pip


> **Colab note:** The `pip` cell above runs without modification in Google Colab; just make sure the repository (or at least `docs/05-llm/`) is uploaded so the PDFs can be read locally.

In [4]:
import json
import math
import os
from pathlib import Path
from typing import List

import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv
from getpass import getpass
from openai import OpenAI
from pydantic import BaseModel, Field, ValidationError, computed_field, field_validator

## API key handling
1. Preferred: store `OPENAI_API_KEY` inside `.env` (never commit it).  
2. If the environment variable is missing (e.g., on Colab), the loader falls back to `getpass`.  
3. For unit tests, you can temporarily uncomment the placeholder line, but do not ship real keys in notebooks.

In [5]:
def load_api_key(env_path: str = ".env") -> str:
    env_file = Path(env_path)
    if env_file.exists():
        load_dotenv(env_file)
    key = os.getenv("OPENAI_API_KEY")
    if not key:
        try:
            key = getpass("Enter your OpenAI API key: ").strip()
        except Exception as exc:
            raise RuntimeError("Unable to capture OPENAI_API_KEY interactively.") from exc
    if not key:
        raise RuntimeError("OPENAI_API_KEY is required. Set it in a .env file or type it interactively.")
    return key

# OPENAI_API_KEY = "sk-proj-example"  # Uncomment only for offline smoke tests
OPENAI_API_KEY = load_api_key()
print(f"Loaded API key prefix: {OPENAI_API_KEY[:8]}******** (masked)")
client = OpenAI(api_key=OPENAI_API_KEY)

Loaded API key prefix: sk-proj-******** (masked)


## Quick sanity check: raw API call
Before diving into engineering workflows, confirm the connection by asking GPT-4o-mini a trivial question.

In [6]:
sanity = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    messages=[
        {"role": "system", "content": "You are a succinct assistant."},
        {"role": "user", "content": "Say hello to the AI in Geotech class in one sentence."},
    ],
)
print(sanity.choices[0].message.content)
if sanity.usage:
    print("Token usage -> input:", sanity.usage.prompt_tokens, "output:", sanity.usage.completion_tokens)

Hello, Geotech class AI!
Token usage -> input: 30 output: 7


## Leaning Tower of Pisa soil narrative
The `gpt-bearing-capacity.pdf` report describes a layered clay profile supporting the circular masonry foundation (20 m diameter, embedded 5 m). We will feed that narrative to GPT for parameter extraction later.

In [10]:
soil_report_text = '''
Estimate the site bearing capacity of a circular foundation with 2m diameter foundation with foundation resting on ground and undrained strength Su is 35 kPa
'''.strip()
print(soil_report_text)

Estimate the site bearing capacity of a circular foundation with 2m diameter foundation with foundation resting on ground and undrained strength Su is 35 kPa


## Simple GPT-4o-mini call (no Pydantic yet)
Ask GPT-4o-mini to outline a conceptual retrofit plan. This still uses the raw chat completion API, but now with an engineering-flavored prompt.

In [11]:
tower_prompt = '''
Estimate the site bearing capacity of a circular foundation with 2m diameter foundation with foundation resting on ground and undrained strength Su is 35 kPa
'''.strip()


simple_completion = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.1,
    messages=[
        {"role": "system", "content": "You are a meticulous geotechnical engineer."},
        {"role": "user", "content": tower_prompt},
    ],
)
print(simple_completion.choices[0].message.content)
if simple_completion.usage:
    print("Token usage -> input:", simple_completion.usage.prompt_tokens,
          "output:", simple_completion.usage.completion_tokens)

To estimate the bearing capacity of a circular foundation resting on cohesive soil (where the undrained shear strength \( S_u \) is given), we can use the following formula for the ultimate bearing capacity \( q_u \) of a shallow foundation:

\[
q_u = c N_c + q N_q + \gamma D N_\gamma
\]

Where:
- \( c \) is the cohesion (in this case, \( S_u \)).
- \( N_c \), \( N_q \), and \( N_\gamma \) are the bearing capacity factors.
- \( q \) is the effective vertical stress at the foundation level (which can be considered as zero for shallow foundations resting on the ground surface).
- \( \gamma \) is the unit weight of the soil (not provided, but can be assumed if needed).
- \( D \) is the depth of the foundation (which is zero for a shallow foundation resting on the ground).

For a circular foundation on cohesive soil, the bearing capacity can be simplified to:

\[
q_u = S_u \cdot N_c
\]

For a circular foundation, the bearing capacity factor \( N_c \) can be approximated as:

\[
N_c \approx